In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import json
import io

from tensorflow.keras import layers
from PIL import Image
from IPython.display import display
from ipywidgets import FileUpload

In [ ]:
train_path = "/kaggle/input/datasets/melikechan/cifar100/cifar100/train"
test_path = "/kaggle/input/datasets/melikechan/cifar100/cifar100/test"

IMG_SIZE = 96
BATCH_SIZE = 64
NUM_CLASSES = 100

In [ ]:
train_data = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True,
    seed=42
)

test_data = tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

In [ ]:
class_names = train_data.class_names

print("Number of classes:", len(class_names))
print(class_names)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_data = train_data.prefetch(AUTOTUNE)
test_data = test_data.prefetch(AUTOTUNE)

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.1)
])

In [ ]:
base_model = tf.keras.applications.EfficientNetV2B0(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

In [ ]:
inputs = layers.Input(
    shape=(IMG_SIZE, IMG_SIZE, 3)
)

x = data_augmentation(inputs)

x = base_model(
    x,
    training=False
)

x = layers.GlobalAveragePooling2D()(x)

x = layers.BatchNormalization()(x)

x = layers.Dense(
    512,
    activation="relu"
)(x)

x = layers.Dropout(0.4)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

model = tf.keras.Model(
    inputs,
    outputs
)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "best_model.keras",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max"
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=8,
        restore_best_weights=True,
        mode="max"
    )
]

In [ ]:
history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=10,
    callbacks=callbacks
)

In [ ]:
base_model.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history_fine = model.fit(
    train_data,
    validation_data=test_data,
    epochs=40,
    callbacks=callbacks
)

In [ ]:
loss, accuracy = model.evaluate(test_data)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)
print("Test Accuracy:", accuracy * 100, "%")

In [ ]:
model = tf.keras.models.load_model(
    "best_model.keras"
)

In [ ]:
loss, accuracy = model.evaluate(test_data)

print("Best Test Loss:", loss)
print("Best Test Accuracy:", accuracy)
print("Best Test Accuracy:", accuracy * 100, "%")

In [ ]:
with open("class_names.json", "w") as f:
    json.dump(class_names, f)

print("Class names saved successfully.")

In [ ]:
upload = FileUpload(
    accept="image/*",
    multiple=False
)

display(upload)

In [ ]:
if len(upload.value) == 0:
    print("Please upload an image first.")
else:
    file_name = list(upload.value.keys())[0]

    image_data = upload.value[file_name]["content"]

    img = Image.open(
        io.BytesIO(image_data)
    ).convert("RGB")

    original_img = img.copy()

    img = img.resize(
        (IMG_SIZE, IMG_SIZE)
    )

    img_array = np.array(img)

    input_image = np.expand_dims(
        img_array,
        axis=0
    )

    prediction = model.predict(
        input_image,
        verbose=0
    )

    predicted_class = np.argmax(
        prediction[0]
    )

    class_name = class_names[
        predicted_class
    ]

    confidence = (
        prediction[0][predicted_class] * 100
    )

    print("Predicted Class:", class_name)
    print(
        "Confidence:",
        f"{confidence:.2f}%"
    )

In [ ]:
if len(upload.value) != 0:

    top_5_indices = np.argsort(
        prediction[0]
    )[-5:][::-1]

    print("\nTop 5 Predictions:")

    for index in top_5_indices:
        print(
            f"{class_names[index]}: "
            f"{prediction[0][index] * 100:.2f}%"
        )

In [ ]:
if len(upload.value) != 0:

    plt.figure(figsize=(6, 6))

    plt.imshow(original_img)

    plt.title(
        f"{class_name} - {confidence:.2f}%"
    )

    plt.axis("off")

    plt.show()

In [ ]:
if len(upload.value) != 0:

    labels = [
        class_names[index]
        for index in top_5_indices
    ]

    values = [
        prediction[0][index] * 100
        for index in top_5_indices
    ]

    plt.figure(figsize=(10, 5))

    plt.barh(
        labels[::-1],
        values[::-1]
    )

    plt.xlabel("Confidence (%)")
    plt.ylabel("Class")
    plt.title("Top 5 Predictions")

    plt.show()